# Predição Final

## Objetivo

Realizar a predição do custo de seguro médico para novos clientes utilizando a abordagem de segmentação por clusters.

O pipeline de predição consiste em:

1. Receber os dados de entrada do novo cliente
2. Aplicar o mesmo encoding e normalização utilizados no treinamento
3. Identificar o cluster ao qual o cliente pertence (usando o K-Means treinado)
4. Aplicar o modelo de regressão específico daquele cluster
5. Retornar o cluster identificado e o custo previsto

Os artefatos utilizados foram salvos nas etapas anteriores:

- `scaler.pkl` → normalizador treinado na Etapa 1
- `kmeans_model.pkl` → modelo K-Means treinado na Etapa 2
- `modelo_cluster_0.pkl` → melhor modelo do Cluster 0 (Etapa 4)
- `modelo_cluster_1.pkl` → melhor modelo do Cluster 1 (Etapa 4)
- `modelo_cluster_2.pkl` → melhor modelo do Cluster 2 (Etapa 4)

In [1]:
import pandas as pd
import numpy as np
import joblib

## Carregamento dos Artefatos

Carregamos os modelos e o normalizador salvos nas etapas anteriores do projeto.

In [2]:
scaler = joblib.load('../data/scaler.pkl')
kmeans = joblib.load('../data/kmeans_model.pkl')

modelos_cluster = {
    0: joblib.load('../data/modelo_cluster_0.pkl'),
    1: joblib.load('../data/modelo_cluster_1.pkl'),
    2: joblib.load('../data/modelo_cluster_2.pkl'),
}

print("Artefatos carregados com sucesso!")
print(f"  Scaler: {type(scaler).__name__}")
print(f"  K-Means: {type(kmeans).__name__} (K={kmeans.n_clusters})")
for k, m in modelos_cluster.items():
    print(f"  Cluster {k}: {type(m).__name__}")

Artefatos carregados com sucesso!
  Scaler: StandardScaler
  K-Means: KMeans (K=3)
  Cluster 0: KNeighborsRegressor
  Cluster 1: KNeighborsRegressor
  Cluster 2: KNeighborsRegressor


## Função de Pré-processamento

Para garantir que os dados de entrada sejam tratados da mesma forma que os dados de treinamento, criamos uma função que:

- Aplica o mesmo encoding das variáveis categóricas (`sex`, `smoker`, `region`)
- Garante que todas as colunas esperadas estejam presentes
- Normaliza com o `StandardScaler` já treinado

> **Importante:** A ordem e os nomes das colunas devem ser idênticos aos utilizados no treinamento:
> `age`, `sex`, `bmi`, `children`, `smoker`, `region_northwest`, `region_southeast`, `region_southwest`

In [3]:
COLUNAS_MODELO = ['age', 'sex', 'bmi', 'children', 'smoker',
                  'region_northwest', 'region_southeast', 'region_southwest']

def preprocessar(age, sex, bmi, children, smoker, region):
    """
    Recebe os dados brutos de um novo cliente e retorna o array
    normalizado pronto para o K-Means e para a regressão.

    Parâmetros
    ----------
    age      : int   — idade do cliente
    sex      : str   — 'male' ou 'female'
    bmi      : float — índice de massa corporal
    children : int   — número de filhos
    smoker   : str   — 'yes' ou 'no'
    region   : str   — 'northeast', 'northwest', 'southeast' ou 'southwest'
    """
    # Encoding de sex e smoker (igual ao LabelEncoder da Etapa 1)
    sex_enc    = 1 if sex.strip().lower() == 'male' else 0        # female=0, male=1
    smoker_enc = 1 if smoker.strip().lower() == 'yes' else 0      # no=0, yes=1

    # One-Hot Encoding de region (drop_first=True elimina 'northeast')
    region = region.strip().lower()
    region_northwest  = 1 if region == 'northwest'  else 0
    region_southeast  = 1 if region == 'southeast'  else 0
    region_southwest  = 1 if region == 'southwest'  else 0

    # Monta o DataFrame com a mesma estrutura do treinamento
    dados = pd.DataFrame([{
        'age'              : age,
        'sex'              : sex_enc,
        'bmi'              : bmi,
        'children'         : children,
        'smoker'           : smoker_enc,
        'region_northwest' : region_northwest,
        'region_southeast' : region_southeast,
        'region_southwest' : region_southwest,
    }])[COLUNAS_MODELO]

    # Normalização com o scaler treinado
    dados_scaled = scaler.transform(dados)
    return dados_scaled

## Função de Predição

Com os dados pré-processados, a função de predição:

1. Identifica o cluster do cliente usando o K-Means
2. Seleciona o modelo de regressão correspondente
3. Retorna o cluster e o custo previsto

In [4]:
def prever(age, sex, bmi, children, smoker, region):
    """
    Realiza a predição completa para um novo cliente.
    Retorna o cluster identificado e o custo previsto.
    """
    # Pré-processamento
    X_novo = preprocessar(age, sex, bmi, children, smoker, region)

    # Identificação do cluster
    cluster = int(kmeans.predict(X_novo)[0])

    # Predição com o modelo especializado do cluster
    modelo = modelos_cluster[cluster]
    custo_previsto = float(modelo.predict(X_novo)[0])

    return cluster, custo_previsto

## Exemplo de Predição

Utilizamos o exemplo fornecido no enunciado da atividade para demonstrar o funcionamento do pipeline.

**Dados de entrada:**

| Campo    | Valor     |
|----------|-----------|
| age      | 40        |
| sex      | male      |
| bmi      | 30.0      |
| children | 2         |
| smoker   | yes       |
| region   | northwest |

In [5]:
# Dados do novo cliente (exemplo do enunciado)
age      = 40
sex      = 'male'
bmi      = 30.0
children = 2
smoker   = 'yes'
region   = 'northwest'

cluster, custo = prever(age, sex, bmi, children, smoker, region)

print("=" * 40)
print("        RESULTADO DA PREDIÇÃO")
print("=" * 40)
print(f"  Cluster identificado : {cluster}")
print(f"  Custo previsto       : US$ {custo:,.2f}")
print("=" * 40)

        RESULTADO DA PREDIÇÃO
  Cluster identificado : 0
  Custo previsto       : US$ 32,238.67


c:\Users\Anna Beatriz\Documents\Faculdade\Machine Learning\OAT2\Code\ML-OAT2-SEGMENTACAO-PREDICAO\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
c:\Users\Anna Beatriz\Documents\Faculdade\Machine Learning\OAT2\Code\ML-OAT2-SEGMENTACAO-PREDICAO\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(


## Múltiplas Predições

É possível realizar predições em lote para vários clientes ao mesmo tempo.

O exemplo abaixo demonstra três perfis distintos de clientes para validar o comportamento do modelo em diferentes cenários.

In [6]:
novos_clientes = [
    {"age": 25, "sex": "female", "bmi": 22.0, "children": 0, "smoker": "no",  "region": "northeast"},
    {"age": 40, "sex": "male",   "bmi": 30.0, "children": 2, "smoker": "yes", "region": "northwest"},
    {"age": 55, "sex": "male",   "bmi": 35.5, "children": 3, "smoker": "no",  "region": "southeast"},
]

resultados = []
for cliente in novos_clientes:
    cluster, custo = prever(**cliente)
    resultados.append({
        **cliente,
        "cluster": cluster,
        "custo_previsto": f"US$ {custo:,.2f}"
    })

df_resultados = pd.DataFrame(resultados)
display(df_resultados)

c:\Users\Anna Beatriz\Documents\Faculdade\Machine Learning\OAT2\Code\ML-OAT2-SEGMENTACAO-PREDICAO\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
c:\Users\Anna Beatriz\Documents\Faculdade\Machine Learning\OAT2\Code\ML-OAT2-SEGMENTACAO-PREDICAO\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
c:\Users\Anna Beatriz\Documents\Faculdade\Machine Learning\OAT2\Code\ML-OAT2-SEGMENTACAO-PREDICAO\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
c:\Users\Anna Beatriz\Documents\Faculdade\Machine Learning\OAT2\Code\ML-OAT2-SEGMENTACAO-PREDICAO\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature 

,age,sex,bmi,children,smoker,region,cluster,custo_previsto
0,25,female,22.0,0,no,northeast,0,"US$ 3,325.42"
1,40,male,30.0,2,yes,northwest,0,"US$ 32,238.67"
2,55,male,35.5,3,no,southeast,2,"US$ 11,992.61"


## Conclusão

O pipeline de predição final integra todas as etapas do projeto:

- O **pré-processamento** garante que os dados novos recebam o mesmo tratamento dos dados de treinamento
- O **K-Means** segmenta o novo cliente no cluster mais adequado ao seu perfil
- O **modelo especializado** do cluster realiza a predição com maior precisão do que um modelo global faria

Essa abordagem é especialmente eficaz para o Cluster 2, onde o KNN especializado alcançou R²=0,9264 — um ganho expressivo em relação ao modelo global (R²=0,8028).